# Initial-margin CVaR MKABSDE

This notebook displays the retained call and butterfly campaigns. It reads the generated JSON files and contains no solver implementation.

## Complete retraining command

Run from the repository root:

```bash
python -m initial_margin.run_campaign \
  --product all \
  --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 \
  --degree 3 \
  --margin-period 0.02 \
  --device cuda
```

The paper defaults use ten independent seeds, 24 Gauss--Legendre nodes for the call tail, and 64 nodes for the complete butterfly law.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
oracle_payload = json.loads((result_root / 'deterministic_oracle.json').read_text())
direct_oracles = {(row['product'], row['time_steps']): row for row in oracle_payload['rows']}

## Initial-margin corrections and numerical variability

The direct column targets the anticipated equation. The non-linear and linear columns target the two proxy BSDEs. To expose the small IM contribution, every entry is $100\,(V_0-V_0^{BS})$, reported as the mean plus or minus the sample standard deviation across seeds.

In [2]:
source_prices = {'call': (2.1959, 2.1971), 'butterfly': (0.3112, 0.3112)}
proxy_oracle_prices = {
    'call': (2.1959478451, 2.1958726630),
    'butterfly': (0.31131, 0.3113085061),
}
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    nonlinear_source, linear_source = source_prices[product]
    nonlinear_oracle, linear_oracle = proxy_oracle_prices[product]
    lines = [
        f'### ATM {product}',
        '',
        '| $N$ | direct MKABSDE | direct oracle | non-linear proxy | linear proxy |',
        '|---:|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {100 * row['direct_mkabsde_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['direct_mkabsde_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * direct_oracles[(product, row['time_steps'])]['price_correction']:.6f} | "
            f"{100 * row['nonlinear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['nonlinear_proxy_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * row['linear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['linear_proxy_price_correction_vs_black_scholes_std']:.2e} |"
        )
    lines.append(
        f'| Independent proxy oracle | — | — | {100 * (nonlinear_oracle - base_price):.6f} | {100 * (linear_oracle - base_price):.6f} |'
    )
    lines.append(
        f'| Source reported value | — | — | {100 * (nonlinear_source - base_price):.6f} | {100 * (linear_source - base_price):.6f} |'
    )
    display(Markdown('\n'.join(lines)))

### ATM call

| $N$ | direct MKABSDE | direct oracle | non-linear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 1.873583 ± 2.79e-03 | 1.873235 | 2.197847 ± 2.85e-03 | 2.190344 ± 2.84e-03 |
| 100 | 1.869610 ± 1.95e-03 | 1.868720 | 2.192509 ± 2.02e-03 | 2.184971 ± 2.01e-03 |
| 200 | 1.866482 ± 2.64e-03 | 1.866000 | 2.188668 ± 2.84e-03 | 2.181119 ± 2.84e-03 |
| Independent proxy oracle | — | — | 2.183615 | 2.176096 |
| Source reported value | — | — | 2.178830 | 2.298830 |

### ATM butterfly

| $N$ | direct MKABSDE | direct oracle | non-linear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.236261 ± 2.00e-04 | 0.236936 | 0.223998 ± 1.41e-04 | 0.223675 ± 1.41e-04 |
| 100 | 0.238475 ± 1.47e-04 | 0.239110 | 0.226270 ± 1.54e-04 | 0.225937 ± 1.54e-04 |
| 200 | 0.239264 ± 2.31e-04 | 0.239904 | 0.227161 ± 1.36e-04 | 0.226823 ± 1.36e-04 |
| Independent proxy oracle | — | — | 0.227529 | 0.227380 |
| Source reported value | — | — | 0.216529 | 0.216529 |

## Interpretation

All IM equations are solved as differences from the analytic no-IM Black--Scholes value, while retaining degree-three Hermite conditional regressions. The analytic value is restored inside the non-linear driver and the CVaR target. The source article has no direct MKABSDE reference; `python -m initial_margin.oracle` supplies an independent one-dimensional state-grid validation without Monte Carlo or regression. For the proxies, the independent call values use the analytic/Feynman--Kac formulas, the linear butterfly uses deterministic quadrature, and the non-linear butterfly uses an independent PDE solve. The source-reported row merely reproduces the rounded or finite-difference numbers printed in that article and is not treated as a high-accuracy oracle. The reported sample standard deviations measure the combined Monte Carlo and regression variability; they are not pure regression-error estimators. Price-level refinement is stable, but it is not by itself a direct test of the full squared-error estimate.